# LARRY normalization comparison

Implements the seven normalization recipes and six measurements
`PROJECT_SUMMARY_27082026.pdf` S3.1 / Table 2 describe, run on the QC'd LARRY
matrix from `larry_preprocessing.ipynb`, and compares the numbers against the
PDF's own reported LARRY values.

Built independently from the PDF's formulas (Appendix B for PFlog) and
standard definitions -- not from any other notebook or the PDF's own
`helpers/` code. The overdispersion estimator in particular is a plain
method-of-moments estimator, **not** a reproduction of the PDF's own
`runorm`-derived estimator, so exact-value agreement isn't expected; the
check here is whether the *ordering/ballpark* of recipes on each measurement
matches, which is what "do the results match up" can mean without their code.

Recipes: `raw`, `lograw`, `log1pCP10k`, `log1pPF`, `PFlog1pPF`, `PFlog`,
`PearsonResid`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)

# [2026-10-01 commented out: /scratch/gzu5140 is purged scratch; see REPOINT_LOG.tsv] PROCESSED = Path("/scratch/gzu5140/ka_twinfer/larry_dataset/processed")
from twinfer.utils.paths import get_larry_dataset_dir  # [2026-10-01 added]
PROCESSED = Path(get_larry_dataset_dir() / "processed")
QC_DIR = PROCESSED / "qc_filtered"

X = sio.mmread(QC_DIR / "larry_qc_counts.mtx").tocsr()
genes = pd.Series(Path(QC_DIR / "genes.txt").read_text().split("\n")).replace("", np.nan).dropna().reset_index(drop=True)
cells_idx = pd.Series(Path(QC_DIR / "cells.txt").read_text().split("\n")).replace("", np.nan).dropna().reset_index(drop=True)
obs = pd.read_csv(QC_DIR / "obs_metadata.csv", index_col=0)
assert X.shape == (len(cells_idx), len(genes))

# day, from the library name (LSK_d2_*, LSK_d4_*, LSK_d6_*)
obs["day"] = obs["library"].str.extract(r"LSK_d(\d+)_").astype(int)
print(X.shape, obs["day"].value_counts().sort_index())


## Overdispersion estimate (per day)

Method-of-moments: for `Var(x_g) = mu_g + alpha * mu_g^2`,
`alpha_g = (var_g - mu_g) / mu_g^2` per gene; take the median over
reasonably-expressed genes as a single per-matrix `alpha`, matching the PDF's
description of measuring `alpha` per matrix rather than assuming a constant.


In [ ]:
def estimate_alpha(Xd):
    Xd = np.asarray(Xd, dtype=float)
    mu = Xd.mean(axis=0)
    var = Xd.var(axis=0, ddof=1)
    ok = mu > 0.1
    alpha_g = np.divide(var[ok] - mu[ok], mu[ok] ** 2, out=np.full(ok.sum(), np.nan), where=mu[ok] ** 2 > 0)
    alpha_g = alpha_g[np.isfinite(alpha_g) & (alpha_g > 0)]
    return float(np.median(alpha_g)) if len(alpha_g) else 0.1


## The seven normalization recipes

In [ ]:
def proportional_fit(Xd, target=None):
    s = Xd.sum(axis=1, keepdims=True)
    s = np.where(s == 0, 1, s)
    if target is None:
        target = np.median(Xd.sum(axis=1))
    return Xd * (target / s)


def recipe_raw(Xd, alpha):
    return Xd.copy()


def recipe_lograw(Xd, alpha):
    return np.log1p(Xd)


def recipe_log1pCP10k(Xd, alpha):
    s = Xd.sum(axis=1, keepdims=True)
    s = np.where(s == 0, 1, s)
    return np.log1p(Xd / s * 1e4)


def recipe_log1pPF(Xd, alpha):
    return np.log1p(proportional_fit(Xd))


def recipe_PFlog1pPF(Xd, alpha):
    return proportional_fit(np.log1p(proportional_fit(Xd)))


def recipe_PFlog(Xd, alpha):
    # z_ci = log1p(4*alpha*x_ci) - rowmean_j log1p(4*alpha*x_cj); K_c*u_ci = 4*alpha*x_ci (cell depth cancels)
    z = np.log1p(4 * alpha * Xd)
    return z - z.mean(axis=1, keepdims=True)


def recipe_PearsonResid(Xd, alpha):
    s = Xd.sum(axis=1, keepdims=True)
    p = Xd.sum(axis=0, keepdims=True) / Xd.sum()
    mu = s * p
    denom = np.sqrt(mu + alpha * mu ** 2)
    denom = np.where(denom == 0, 1, denom)
    return (Xd - mu) / denom


RECIPES = {
    "raw": recipe_raw,
    "lograw": recipe_lograw,
    "log1pCP10k": recipe_log1pCP10k,
    "log1pPF": recipe_log1pPF,
    "PFlog1pPF": recipe_PFlog1pPF,
    "PFlog": recipe_PFlog,
    "PearsonResid": recipe_PearsonResid,
}

# Analytic q_g = (h'(mu))^2 * (mu + alpha*mu^2) for recipes of the shifted-log form
# z = log(y + s*/K). K differs by recipe and must be scaled against the MEAN per-cell
# depth s_star specifically -- log1pCP10k fixes K=1e4, PFlog fixes K=4*alpha*s_star,
# lograw is K=s_star -- while log1pPF's own PF step targets the MEDIAN depth, so its K
# is median_depth even though the h' formula still divides by s_star. A prior version
# used median depth in place of s_star for log1pCP10k, and silently reused lograw's h'
# for log1pPF (i.e. treated its K as equal to s_star); both are wrong whenever mean !=
# median depth, which is always true here since per-cell depth is right-skewed. PDF
# leaves raw/PFlog1pPF blank (outside the shifted-log family) and PearsonResid's column 1
# equals its column 4 (empirical substitute) by definition -- both handled in measure_recipe.
def scale_K(name, s_star, median_depth, alpha):
    if name == "lograw":
        return s_star
    if name == "log1pCP10k":
        return 1e4
    if name == "log1pPF":
        return median_depth
    if name == "PFlog":
        return 4 * alpha * s_star
    return None  # raw, PFlog1pPF: outside the shifted-log family


def h_prime_shifted_log(K, s_star, mu):
    return (K / s_star) / (1.0 + mu * K / s_star)


## The six measurements (PDF Table 2 columns)

In [ ]:
def within_cell_rank_corr(raw, z, n_sample_cells=500):
    idx = rng.choice(raw.shape[0], size=min(n_sample_cells, raw.shape[0]), replace=False)
    rhos = [spearmanr(raw[i], z[i]).correlation for i in idx]
    rhos = [r for r in rhos if np.isfinite(r)]
    return float(np.mean(rhos)) if rhos else np.nan


def depth_r2(y, X_design):
    # R^2 of y regressed on X_design (adds intercept), via lstsq -- avoids an sklearn dependency here
    A = np.column_stack([np.ones(len(y)), X_design])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ coef
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan


def measure_recipe(name, fn, Xd, alpha, raw_depth):
    z = fn(Xd, alpha)

    mu = Xd.mean(axis=0)
    s_star = float(raw_depth.mean())
    median_depth = float(np.median(raw_depth[raw_depth > 0]))
    K = scale_K(name, s_star, median_depth, alpha)
    if K is not None:
        q = h_prime_shifted_log(K, s_star, mu) ** 2 * (mu + alpha * mu ** 2)
        pred_tech_var_spread = float(np.std(q) / np.mean(q)) if np.mean(q) != 0 else np.nan
    elif name == "PearsonResid":
        pred_tech_var_spread = None  # filled below, equals measured gene variance spread
    else:
        pred_tech_var_spread = np.nan  # raw, PFlog1pPF: outside the shifted-log family (PDF leaves blank)

    z_depth = z.sum(axis=1)
    transformed_vs_raw_depth = depth_r2(z_depth, raw_depth.reshape(-1, 1))

    within_cell_ranks = within_cell_rank_corr(Xd, z)

    gene_var = z.var(axis=0, ddof=1)
    measured_gene_var_spread = float(np.std(gene_var) / np.mean(gene_var)) if np.mean(gene_var) != 0 else np.nan
    if pred_tech_var_spread is None:
        pred_tech_var_spread = measured_gene_var_spread  # PearsonResid: PDF's stated empirical substitute

    n_pcs = min(50, z.shape[0] - 1, z.shape[1] - 1)
    pcs = PCA(n_components=n_pcs, random_state=0).fit_transform(z - z.mean(axis=0))
    depth_from_components = depth_r2(raw_depth, pcs)

    n_sub = min(2000, z.shape[0])
    sub_idx = rng.choice(z.shape[0], size=n_sub, replace=False)
    emb = pcs[sub_idx]
    dep = raw_depth[sub_idx]
    pair_i = rng.integers(0, n_sub, size=20000)
    pair_j = rng.integers(0, n_sub, size=20000)
    keep_pairs = pair_i != pair_j
    pair_i, pair_j = pair_i[keep_pairs], pair_j[keep_pairs]
    dist = np.linalg.norm(emb[pair_i] - emb[pair_j], axis=1)
    depth_diff = np.abs(dep[pair_i] - dep[pair_j])
    distance_vs_depth_diff = depth_r2(dist, depth_diff.reshape(-1, 1))

    return {
        "recipe": name,
        "predicted_technical_variance_spread": pred_tech_var_spread,
        "transformed_vs_raw_depth_R2": transformed_vs_raw_depth,
        "within_cell_ranks": within_cell_ranks,
        "measured_gene_variance_spread": measured_gene_var_spread,
        "depth_from_components_R2": depth_from_components,
        "distance_vs_depth_difference_R2": distance_vs_depth_diff,
    }


## Run all seven recipes, per day

In [ ]:
N_HVG = 2000  # restrict to the top variable genes so PCA/pairwise-distance steps stay tractable

results = []
for day in sorted(obs["day"].unique()):
    day_mask = np.flatnonzero((obs["day"] == day).to_numpy())
    Xd_full = np.asarray(X[day_mask].todense(), dtype=float)
    raw_depth = Xd_full.sum(axis=1)

    gene_var_raw = Xd_full.var(axis=0)
    hvg_idx = np.argsort(gene_var_raw)[::-1][:N_HVG]
    Xd = Xd_full[:, hvg_idx]

    alpha = estimate_alpha(Xd)
    print(f"day {day}: {Xd.shape[0]} cells, alpha={alpha:.4f}")

    for name, fn in RECIPES.items():
        row = measure_recipe(name, fn, Xd, alpha, raw_depth)
        row["day"] = day
        results.append(row)

results_df = pd.DataFrame(results).set_index(["day", "recipe"])
results_df.round(4)


## Compare against `PROJECT_SUMMARY_27082026.pdf` Table 2 (LARRY)

Reference values transcribed directly from the PDF's Table 2 (`predicted
technical variance, spread` and `transformed vs. raw depth` columns; the PDF's
other three measurements differ enough in protocol -- e.g. its `distance vs.
depth difference` samples 1,000 cells in gene space rather than 2,000 in a
50-PC embedding -- that a numeric side-by-side would compare different
quantities, so only these two are checked directly here).


In [ ]:
pdf_table2 = pd.DataFrame([
    # day, recipe, predicted_technical_variance_spread, transformed_vs_raw_depth_R2
    (2, "log1pCP10k", 1.015, 0.853), (4, "log1pCP10k", 1.118, 0.834), (6, "log1pCP10k", 1.244, 0.813),
    (2, "log1pPF", 1.648, 0.741), (4, "log1pPF", 1.693, 0.722), (6, "log1pPF", 1.876, 0.703),
    (2, "PFlog", 1.386, 0.003), (4, "PFlog", 1.406, 0.000), (6, "PFlog", 1.417, 0.012),
    (2, "PearsonResid", 0.707, 0.001), (4, "PearsonResid", 0.856, 0.009), (6, "PearsonResid", 1.275, 0.001),
    (2, "lograw", 1.564, 0.989), (4, "lograw", 1.637, 0.987), (6, "lograw", 1.786, 0.984),
], columns=["day", "recipe", "pdf_predicted_technical_variance_spread", "pdf_transformed_vs_raw_depth_R2"]
).set_index(["day", "recipe"])

comparison = results_df.join(pdf_table2, how="left")
comparison[[
    "predicted_technical_variance_spread", "pdf_predicted_technical_variance_spread",
    "transformed_vs_raw_depth_R2", "pdf_transformed_vs_raw_depth_R2",
]].round(4)


In [ ]:
# Ranking check: does our run agree with the PDF on which recipe scores lowest
# (best) on each of these two measurements, per day? (Lower is better on both.)
for day in sorted(obs["day"].unique()):
    ours = results_df.loc[day, "transformed_vs_raw_depth_R2"].dropna()
    pdf = pdf_table2.loc[day, "pdf_transformed_vs_raw_depth_R2"].dropna()
    common = ours.index.intersection(pdf.index)
    our_best = ours[common].idxmin()
    pdf_best = pdf[common].idxmin()
    print(f"day {day}: our best (transformed vs raw depth) = {our_best!r}, "
          f"PDF's best = {pdf_best!r}, {'MATCH' if our_best == pdf_best else 'DIFFERS'}")
